# Interpretable Models — Facility Capacity

Readable counterparts to the MLP: **Ridge regression** (coefficients ≈ multiplicative effect on capacity) and a **shallow decision tree** (explicit split rules), plus permutation importance. Uses the same 80/10/10 split recipe and test metrics as the training pipeline notebook. Check that the test rows match before comparing scores.

Uses the enriched table if present (run Part 2 of the training pipeline notebook first to create it); otherwise falls back to facilities-only features. Runs on plain serverless — no GPU needed.

## Weights & Biases setup

Open this notebook from the repository Git folder so it can import the adjacent `wandb_models.py`. Install the logging and serialization packages below. Set `WANDB_PROJECT` to choose a project. Runs default to offline. To log online, set `WANDB_MODE=online` and configure `WANDB_API_KEY` in your environment or workspace secrets.

The saved models include the fitted preprocessing and target transform. Their `predict()` method returns beds directly.

In [ ]:
%pip install wandb skops

In [0]:
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor, plot_tree

CATALOG = "databricks_virtue_foundation_dataset_dais_2026"
SCHEMA = "virtue_foundation_dataset"
TBL_FACILITIES = f"{CATALOG}.{SCHEMA}.facilities"
ENRICHED_TABLE = "workspace.default.facilities_enriched"

TARGET = "capacity"
RANDOM_STATE = 42

NFHS_FEATURES = [
    "women_age_15_49_with_10_or_more_years_of_schooling_pct",
    "population_below_age_15_years_pct",
    "hh_use_improved_sanitation_pct",
    "women_age_15_49_years_whose_bmi_bmi_is_underweight_bmi_lt_1_pct",
    "hh_electricity_pct",
    "child_u5_who_are_stunted_height_for_age_18_pct",
    "households_using_clean_fuel_for_cooking_pct",
    "w20_24_married_before_age_18_years_pct",
    "female_population_age_6_years_and_above_ever_schooled_pct",
    "hh_member_covered_health_insurance_pct",
    "hh_improved_water_pct",
    "all_w15_49_who_are_anaemic_pct",
    "sex_ratio_total_f_per_1000_m",
    "households_using_iodized_salt_pct",
    "births_attended_by_skilled_hp_5y_10_pct",
]

NUMERIC_FEATURES = [
    "numberDoctors",
    "yearEstablished",
    "latitude",
    "longitude",
    "engagement_metrics_n_followers",
    "engagement_metrics_n_likes",
    "engagement_metrics_n_engagements",
    "post_metrics_post_count",
    "number_of_facts_about_the_organization",
    "distinct_social_media_presence_count",
] + NFHS_FEATURES

CATEGORICAL_FEATURES = [
    "facilityTypeId",
    "operatorTypeId",
    "address_stateOrRegion",
    "organization_type",
    "affiliated_staff_presence",
    "custom_logo_presence",
]

def to_na(s):
    return s.astype("string").str.strip().str.lower().where(
        lambda x: ~x.isin({"null", "none", "nan", "n/a", "na", "", "-", "unknown"}), other=pd.NA
    )

def regression_report(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    print(f"{name}: MAE {mae:.2f} | RMSE {rmse:.2f} | R2 {r2:.3f}")
    return {"mae": mae, "rmse": rmse, "r2": r2}

In [0]:
# Uses the enriched table if present (created by Part 2 of the training
# pipeline notebook); otherwise falls back to facilities-only features.
if spark.catalog.tableExists(ENRICHED_TABLE):
    raw = spark.table(ENRICHED_TABLE).toPandas()
    print(f"loaded enriched table: {len(raw)} rows (incl. {len(NFHS_FEATURES)} NFHS district features)")
else:
    raw = spark.table(TBL_FACILITIES).toPandas()
    NUMERIC_FEATURES = [c for c in NUMERIC_FEATURES if c not in NFHS_FEATURES]
    print("enriched table not found -> facilities-only features (run Part 2 of the training pipeline notebook to add NFHS features)")

facilities = raw.copy()
for col in [TARGET] + [c for c in NUMERIC_FEATURES if c not in NFHS_FEATURES]:
    facilities[col] = pd.to_numeric(to_na(facilities[col]), errors="coerce")
for col in CATEGORICAL_FEATURES:
    facilities[col] = to_na(facilities[col]).astype("object")
if "d" in facilities.columns:  # enriched frame carries district features
    for col in NFHS_FEATURES:
        facilities[col] = pd.to_numeric(facilities[col], errors="coerce")

# Text signal (same verified screening as the other notebooks).
TEXT_FEATURES = ["equipment", "capability", "procedure", "specialties", "affiliationTypeIds"]
for col in TEXT_FEATURES:
    facilities[f"{col}_len"] = to_na(raw[col]).str.len().fillna(0)
    facilities[f"{col}_present"] = to_na(raw[col]).notna().astype(int)
NUMERIC_FEATURES = NUMERIC_FEATURES + [f"{c}_len" for c in TEXT_FEATURES] + [f"{c}_present" for c in TEXT_FEATURES]

labeled = facilities.dropna(subset=[TARGET]).copy()
print(f"rows: {len(facilities)} | with a capacity label: {len(labeled)}")
print(f"labeled rows with NFHS district features: {labeled[NFHS_FEATURES[0]].notna().sum() if 'd' in facilities.columns else 0}")
display(labeled[TARGET].describe().to_frame())

In [0]:
# Split: same shuffled random split (80/10/10, seed 42) as the other notebooks
# so metrics are directly comparable.
X = labeled[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = labeled[TARGET]

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=RANDOM_STATE)

print(f"train: {len(X_train)} | val: {len(X_val)} | test: {len(X_test)}")

In [0]:
# Ridge on log1p(capacity). The target transform lives inside
# TransformedTargetRegressor, so predict() returns original units.
preprocessor = ColumnTransformer(
    [
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC_FEATURES),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5))]), CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)

ridge_model = TransformedTargetRegressor(
    regressor=Pipeline([("preprocessor", preprocessor), ("ridge", Ridge(alpha=1.0))]),
    func=np.log1p, inverse_func=np.expm1,
)
ridge_model.fit(X_train, y_train)
ridge_pred = ridge_model.predict(X_test)
ridge_metrics = regression_report(y_test, ridge_pred, "Ridge (log target)")

# Coefficients -> approximate multiplicative effect on capacity: exp(coef)-1.
# For one-hot columns that means 'attribute present vs absent'; for
# standardized numerics, 'per one standard deviation'.
ridge_pipe = ridge_model.regressor_
coefs = ridge_pipe.named_steps["ridge"].coef_
feat_names = [n.split("__", 1)[-1] for n in ridge_pipe.named_steps["preprocessor"].get_feature_names_out()]
coef_df = (pd.DataFrame({"feature": feat_names, "coef": coefs, "effect_pct": np.expm1(coefs) * 100})
           .iloc[np.argsort(-np.abs(coefs))]
           .head(15)
           .reset_index(drop=True))
display(coef_df)

In [0]:
# Shallow decision tree: explicit, quotable split rules on log1p(capacity).
tree_model = TransformedTargetRegressor(
    regressor=Pipeline([("preprocessor", preprocessor), ("tree", DecisionTreeRegressor(max_depth=4, min_samples_leaf=25, random_state=RANDOM_STATE))]),
    func=np.log1p, inverse_func=np.expm1,
)
tree_model.fit(X_train, y_train)
tree_pred = tree_model.predict(X_test)
tree_metrics = regression_report(y_test, tree_pred, "Decision tree (depth 4)")

tree_pipeline = tree_model.regressor_
tree = tree_pipeline.named_steps["tree"]
feat_names = [n.split("__", 1)[-1] for n in tree_pipeline.named_steps["preprocessor"].get_feature_names_out()]

plt.figure(figsize=(24, 10))
plot_tree(tree, feature_names=feat_names, filled=True, fontsize=7, impurity=False)
plt.title("Decision tree for log1p(capacity)")
plt.show()

imp_df = (pd.DataFrame({"feature": feat_names, "importance": tree.feature_importances_})
          .query("importance > 0")
          .sort_values("importance", ascending=False)
          .reset_index(drop=True))
display(imp_df)

In [0]:
# Which raw features does the Ridge model actually rely on? Test-set
# permutation importance: mean MAE increase when a feature is shuffled.
perm = permutation_importance(ridge_model, X_test, y_test, scoring="neg_mean_absolute_error", n_repeats=10, random_state=RANDOM_STATE)
perm_df = (pd.DataFrame({"feature": X_test.columns, "importance": perm.importances_mean})
           .sort_values("importance", ascending=False)
           .head(15)
           .reset_index(drop=True))

fig, ax = plt.subplots(figsize=(8, 6))
perm_df.plot(kind="barh", x="feature", y="importance", ax=ax, legend=False)
ax.set_title("Permutation importance (mean MAE increase when shuffled)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()
display(perm_df)

In [0]:
# Save the exact fitted models used for the test metrics.
from pathlib import Path
import sys

# Supports running from a Git-folder notebook or the checkout root.
notebooks_folder = Path("ML/notebooks").resolve()
if (notebooks_folder / "wandb_models.py").is_file() and str(notebooks_folder) not in sys.path:
    sys.path.insert(0, str(notebooks_folder))

from wandb_models import log_sklearn_capacity

feature_set = "nfhs-enriched" if any(field in NFHS_FEATURES for field in NUMERIC_FEATURES) else "base"
training_config = {
    "target": TARGET,
    "feature_set": feature_set,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "split_seed": RANDOM_STATE,
    "n_train": len(X_train),
    "n_val": len(X_val),
    "n_test": len(X_test),
}
ridge_artifact = log_sklearn_capacity(
    name=f"capacity-ridge-{feature_set}",
    model=ridge_model,
    metrics={f"test_{metric}": value for metric, value in ridge_metrics.items()},
    config={**training_config, "alpha": 1.0},
    target_transform="log1p/expm1",
)
tree_artifact = log_sklearn_capacity(
    name=f"capacity-tree-{feature_set}",
    model=tree_model,
    metrics={f"test_{metric}": value for metric, value in tree_metrics.items()},
    config={**training_config, "max_depth": 4, "min_samples_leaf": 25},
    target_transform="log1p/expm1",
)
print("Ridge:", ridge_artifact)
print("Decision tree:", tree_artifact)


## Reading the results

* `effect_pct` in the Ridge table ≈ percent change in capacity associated with the feature (one SD for standardized numerics, presence vs absence for one-hot columns). Expect operator/facility type and doctor count to dominate.
* The tree gives explicit threshold rules (e.g. numberDoctors > N → higher-capacity leaf) — easy to quote in a write-up.
* Compare test MAE with the baseline and MLP when the same facilities appear in each test set. Similar error can favor the simpler model.
* Use the saved model artifact and its ordered feature schema to reproduce predictions in beds.

# Interpretable Models — Facility Capacity

**Goal:** complement the MLP with models you can actually read and quote in a write-up:

* **Ridge regression** — coefficients translate into approximate multiplicative effects on capacity (e.g. "public facilities are associated with ~X% more capacity, all else equal").
* **Shallow decision tree** — explicit threshold rules (``numberDoctors > N`` → higher-capacity leaf), plottable and explainable.
* **Permutation importance** — which raw features the model actually relies on.

Uses the same 80/10/10 split recipe and test metrics as the other notebooks. Check the test facility IDs before comparing scores. If the enriched table exists (created in Part 2 of the *Enriched Capacity GPU Training* notebook), the NFHS-5 district features are included automatically. Otherwise it uses facility features.

Runs on plain serverless; no GPU needed.

In [0]:
# Setup: shared config with the other capacity notebooks.
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor, plot_tree

CATALOG = "databricks_virtue_foundation_dataset_dais_2026"
SCHEMA = "virtue_foundation_dataset"
TBL_FACILITIES = f"{CATALOG}.{SCHEMA}.facilities"
ENRICHED_TABLE = "workspace.default.facilities_enriched"

TARGET = "capacity"
RANDOM_STATE = 42

# 15 NFHS-5 district indicators (selected in the enrichment part of the
# 'Enriched Capacity GPU Training' notebook).
NFHS_FEATURES = [
    "women_age_15_49_with_10_or_more_years_of_schooling_pct",
    "population_below_age_15_years_pct",
    "hh_use_improved_sanitation_pct",
    "women_age_15_49_years_whose_bmi_bmi_is_underweight_bmi_lt_1_pct",
    "hh_electricity_pct",
    "child_u5_who_are_stunted_height_for_age_18_pct",
    "households_using_clean_fuel_for_cooking_pct",
    "w20_24_married_before_age_18_years_pct",
    "female_population_age_6_years_and_above_ever_schooled_pct",
    "hh_member_covered_health_insurance_pct",
    "hh_improved_water_pct",
    "all_w15_49_who_are_anaemic_pct",
    "sex_ratio_total_f_per_1000_m",
    "households_using_iodized_salt_pct",
    "births_attended_by_skilled_hp_5y_10_pct",
]

NUMERIC_FEATURES = [
    "numberDoctors",
    "yearEstablished",
    "latitude",
    "longitude",
    "engagement_metrics_n_followers",
    "engagement_metrics_n_likes",
    "engagement_metrics_n_engagements",
    "post_metrics_post_count",
    "number_of_facts_about_the_organization",
    "distinct_social_media_presence_count",
] + NFHS_FEATURES

CATEGORICAL_FEATURES = [
    "facilityTypeId",
    "operatorTypeId",
    "address_stateOrRegion",
    "organization_type",
    "affiliated_staff_presence",
    "custom_logo_presence",
]

def to_num(s):
    m = re.match(r"^\s*-?\d+(\.\d+)?\s*$", str(s))
    return float(m.group(0)) if m else np.nan

def to_na(s):
    return s.astype("string").str.strip().str.lower().where(
        lambda x: ~x.isin({"null", "none", "nan", "n/a", "na", "", "-", "unknown"}), other=pd.NA
    )

def regression_report(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    print(f"{name}: MAE {mae:.2f} | RMSE {rmse:.2f} | R2 {r2:.3f}")
    return {"mae": mae, "rmse": rmse, "r2": r2}

In [0]:
# Load and prepare. Uses the enriched table if present; otherwise falls back
# to facilities-only features.
if spark.catalog.tableExists(ENRICHED_TABLE):
    raw = spark.table(ENRICHED_TABLE).toPandas()
    print(f"loaded enriched table: {len(raw)} rows (incl. {len(NFHS_FEATURES)} NFHS district features)")
else:
    raw = spark.table(TBL_FACILITIES).toPandas()
    print("enriched table not found -> facilities-only features")
    NUMERIC_FEATURES = [c for c in NUMERIC_FEATURES if c not in NFHS_FEATURES]

facilities = raw.copy()
for col in [c for c in NUMERIC_FEATURES if c not in NFHS_FEATURES] + [TARGET]:
    facilities[col] = pd.to_numeric(to_na(facilities[col]), errors="coerce")
for col in CATEGORICAL_FEATURES:
    facilities[col] = to_na(facilities[col]).astype("object")
if "d" in facilities.columns:  # enriched frame carries district features
    for col in NFHS_FEATURES:
        facilities[col] = pd.to_numeric(facilities[col], errors="coerce")

# Free-text signal (same screening as the other notebooks): length/presence of
# these scraped columns correlates with capacity.
TEXT_FEATURES = ["equipment", "capability", "procedure", "specialties", "affiliationTypeIds"]
for col in TEXT_FEATURES:
    facilities[f"{col}_len"] = to_na(raw[col]).str.len().fillna(0)
    facilities[f"{col}_present"] = to_na(raw[col]).notna().astype(int)
NUMERIC_FEATURES = NUMERIC_FEATURES + [f"{c}_len" for c in TEXT_FEATURES] + [f"{c}_present" for c in TEXT_FEATURES]

labeled = facilities.dropna(subset=[TARGET]).copy()
print(f"rows: {len(facilities)} | with a capacity label: {len(labeled)}")
display(labeled[TARGET].describe().to_frame())

In [0]:
# Same 80/10/10 shuffled split as the other notebooks (rows are independent
# facilities; no event timestamp).
X = labeled[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = labeled[TARGET]

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=RANDOM_STATE)

print(f"train: {len(X_train)} | val: {len(X_val)} | test: {len(X_test)}")

In [0]:
# Ridge regression on log1p(capacity). The target transform lives inside the
# model (TransformedTargetRegressor) so predict() returns original units and
# the logged artifact is deployable as-is.
preprocessor = ColumnTransformer(
    [
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC_FEATURES),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5))]), CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)

ridge_model = TransformedTargetRegressor(
    regressor=Pipeline([("preprocessor", preprocessor), ("ridge", Ridge(alpha=1.0))]),
    func=np.log1p, inverse_func=np.expm1,
)
ridge_model.fit(X_train, y_train)
ridge_pred = ridge_model.predict(X_test)
ridge_metrics = regression_report(y_test, ridge_pred, "Ridge (log target)")

# Coefficients -> approximate multiplicative effect on capacity: exp(coef)-1.
# For one-hot columns: having the attribute vs not. For standardized numerics:
# per one SD increase.
coefs = ridge_model.regressor_.named_steps["ridge"].coef_
feat_names = [n.split("__", 1)[-1] for n in ridge_model.regressor_.named_steps["preprocessor"].get_feature_names_out()]
coef_df = (pd.DataFrame({"feature": feat_names, "coef": coefs, "effect_pct": np.expm1(coefs) * 100})
           .iloc[np.argsort(-np.abs(coefs))]
           .head(15)
           .reset_index(drop=True))
display(coef_df)

In [0]:
# Shallow decision tree: explicit, quotable threshold rules.
tree_model = TransformedTargetRegressor(
    regressor=Pipeline([
        ("preprocessor", preprocessor),
        ("tree", DecisionTreeRegressor(max_depth=4, min_samples_leaf=25, random_state=RANDOM_STATE)),
    ]),
    func=np.log1p, inverse_func=np.expm1,
)
tree_model.fit(X_train, y_train)
tree_pred = tree_model.predict(X_test)
tree_metrics = regression_report(y_test, tree_pred, "Decision tree (depth 4)")

tree_pipeline = tree_model.regressor_
tree = tree_pipeline.named_steps["tree"]
tree_names = [n.split("__", 1)[-1] for n in tree_pipeline.named_steps["preprocessor"].get_feature_names_out()]

plt.figure(figsize=(24, 10))
plot_tree(tree, feature_names=tree_names, filled=True, fontsize=7, impurity=False)
plt.title("Decision tree for log1p(capacity) — read thresholds top to bottom")
plt.show()

imp_df = (pd.DataFrame({"feature": tree_names, "importance": tree.feature_importances_})
          .sort_values("importance", ascending=False)
          .head(10)
          .reset_index(drop=True))
display(imp_df)

In [0]:
# Which raw features does the Ridge model actually rely on? Test-set
# permutation importance: how much MAE increases when a feature is shuffled.
perm = permutation_importance(ridge_model, X_test, y_test,
                               scoring="neg_mean_absolute_error",
                               n_repeats=10, random_state=RANDOM_STATE)
perm_df = (pd.DataFrame({"feature": X_test.columns, "importance": perm.importances_mean})
           .sort_values("importance", ascending=False)
           .head(15)
           .reset_index(drop=True))

fig, ax = plt.subplots(figsize=(8, 6))
perm_df.plot(kind="barh", x="feature", y="importance", ax=ax, legend=False)
ax.invert_yaxis()
ax.set_title("Permutation importance (mean MAE increase when shuffled)")
ax.set_xlabel("MAE increase")
plt.tight_layout()
plt.show()
display(perm_df)

In [0]:
# Save the exact fitted models used for the test metrics.
from pathlib import Path
import sys

# Supports running from a Git-folder notebook or the checkout root.
notebooks_folder = Path("ML/notebooks").resolve()
if (notebooks_folder / "wandb_models.py").is_file() and str(notebooks_folder) not in sys.path:
    sys.path.insert(0, str(notebooks_folder))

from wandb_models import log_sklearn_capacity

feature_set = "nfhs-enriched" if any(field in NFHS_FEATURES for field in NUMERIC_FEATURES) else "base"
training_config = {
    "target": TARGET,
    "feature_set": feature_set,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "split_seed": RANDOM_STATE,
    "n_train": len(X_train),
    "n_val": len(X_val),
    "n_test": len(X_test),
}
ridge_artifact = log_sklearn_capacity(
    name=f"capacity-ridge-{feature_set}",
    model=ridge_model,
    metrics={f"test_{metric}": value for metric, value in ridge_metrics.items()},
    config={**training_config, "alpha": 1.0},
    target_transform="log1p/expm1",
)
tree_artifact = log_sklearn_capacity(
    name=f"capacity-tree-{feature_set}",
    model=tree_model,
    metrics={f"test_{metric}": value for metric, value in tree_metrics.items()},
    config={**training_config, "max_depth": 4, "min_samples_leaf": 25},
    target_transform="log1p/expm1",
)
print("Ridge:", ridge_artifact)
print("Decision tree:", tree_artifact)


## Reading the results

* **Ridge `effect_pct`** = approximate % change in capacity associated with a feature (one SD for standardized numeric features; presence vs absence for one-hot columns). Expect operator type (public), facility type (hospital) and doctor count to dominate — consistent with the screening correlations.
* **The tree** gives explicit threshold rules (e.g. `numberDoctors > N` → higher-capacity leaf) that are easy to quote in the write-up.
* **Compare test MAE** with the HistGradientBoosting baseline and MLP when the same facilities appear in each test set. Similar error can favor the simpler model.
* Use the saved model artifact and its ordered feature schema to reproduce predictions in beds.